<a href="https://colab.research.google.com/github/aparna-2001/GATE_PPI/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import pickle
import json
import pandas as pd

base_dir = '/content/drive/MyDrive/ppi_gnn'

# 1. The final pairs file — protein1, protein2, label, split for every pair
pairs_final_aligned = pd.read_parquet(f'{base_dir}/data/labels/pairs_final_aligned.parquet')
print(f"Pairs loaded: {len(pairs_final_aligned):,}")

# 2. The actual graph objects — one per protein, the real model input
#    (these are already saved as individual .pt files, loaded on demand,
#    not all at once — see below)
import os
graph_dir = f'{base_dir}/data/processed/graphs'
print(f"Graph files available: {len(os.listdir(graph_dir)):,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pairs loaded: 5,016
Graph files available: 1,758


full_backbone_checkpoint.json     → already baked into the graphs, not needed directly
final_node_features.pkl           → already baked into the graphs, not needed directly
all_edges.pkl / all_edge_features → already baked into the graphs, not needed directly
esm_embeddings_step8d.pkl         → already baked into the graphs, not needed directly
structure_data_final.json         → only needed if you want to inspect source (PDB/AF2/ESMFold)
                                      per protein, but this is also stored on each graph object
                                      itself (graph.source), so not strictly required either

In [ ]:
!pip install torch_geometric --quiet

In [ ]:
import torch

sample_uid = pairs_final_aligned.iloc[0]['protein1']
sample_graph = torch.load(f'{graph_dir}/{sample_uid}.pt', weights_only=False)
print(sample_graph)
print(f"Source: {sample_graph.source}")

Data(x=[1139, 517], edge_index=[2, 6127], edge_attr=[6127, 25], pos=[1139, 3], edge_vector=[6127, 3], edge_type=[6127], uniprot_id='P13569', n_residues=1139, source='PDB')
Source: PDB


In [ ]:
import torch
print(f"Sample node feature dim: {sample_graph.x.shape[1]}")
print(f"Does graph have a vector node feature? {'pos' in sample_graph.keys() if hasattr(sample_graph, 'keys') else 'pos' in dir(sample_graph)}")

Sample node feature dim: 517
Does graph have a vector node feature? True


In [ ]:
import torch
import torch.nn as nn

class GVP(nn.Module):
    """
    Geometric Vector Perceptron (Jing et al., 2021).
    Takes (scalar, vector) features, returns (scalar, vector) features.
    The vector path only ever applies rotation-equivariant linear maps and
    gating by scalars (never directly mixes vector components nonlinearly),
    which is what preserves equivariance.
    """
    def __init__(self, in_dims, out_dims, activation=nn.ReLU()):
        super().__init__()
        self.si, self.vi = in_dims
        self.so, self.vo = out_dims

        self.h_dim = max(self.vi, self.vo)
        self.wh = nn.Linear(self.vi, self.h_dim, bias=False)
        self.ws = nn.Linear(self.h_dim + self.si, self.so)
        self.wv = nn.Linear(self.h_dim, self.vo, bias=False)

        self.activation = activation
        self.vector_gate = nn.Linear(self.so, self.vo)

    def forward(self, s, v):
        vh = self.wh(v.transpose(-1, -2)).transpose(-1, -2)
        vn = torch.norm(vh, dim=-1)

        s_combined = torch.cat([s, vn], dim=-1)
        s_out = self.ws(s_combined)
        s_out = self.activation(s_out)

        v_out = self.wv(vh.transpose(-1, -2)).transpose(-1, -2)
        gate = torch.sigmoid(self.vector_gate(s_out)).unsqueeze(-1)
        v_out = v_out * gate

        return s_out, v_out


# Quick test on dummy data
N = 10
gvp = GVP(in_dims=(128, 4), out_dims=(128, 16))
s_test = torch.randn(N, 128)
v_test = torch.randn(N, 4, 3)
s_o, v_o = gvp(s_test, v_test)
print(f"Input:  s={s_test.shape}, v={v_test.shape}")
print(f"Output: s={s_o.shape}, v={v_o.shape}")

Input:  s=torch.Size([10, 128]), v=torch.Size([10, 4, 3])
Output: s=torch.Size([10, 128]), v=torch.Size([10, 16, 3])


In [ ]:
def random_rotation_matrix():
    """Generate a random 3D rotation matrix via QR decomposition."""
    A = torch.randn(3, 3)
    Q, R = torch.linalg.qr(A)
    # Ensure proper rotation (det=+1, not a reflection)
    if torch.det(Q) < 0:
        Q[:, 0] = -Q[:, 0]
    return Q

torch.manual_seed(42)
R = random_rotation_matrix()
print(f"Rotation matrix determinant (should be ~1.0): {torch.det(R):.4f}")

# Run the same GVP on original vectors, and on rotated vectors
gvp.eval()  # disable any randomness e.g. dropout, though we have none here

with torch.no_grad():
    s_orig, v_orig = gvp(s_test, v_test)

    v_rotated_input = v_test @ R.T   # rotate input vectors
    s_rot, v_rot_output = gvp(s_test, v_rotated_input)

    # Check: does rotating the input and THEN running GVP equal
    # running GVP and THEN rotating the output?
    v_orig_then_rotated = v_orig @ R.T

    scalar_diff = (s_orig - s_rot).abs().max().item()
    vector_diff = (v_orig_then_rotated - v_rot_output).abs().max().item()

print(f"\nMax difference in scalar outputs (should be ~0): {scalar_diff:.6f}")
print(f"Max difference in vector outputs (should be ~0): {vector_diff:.6f}")

Rotation matrix determinant (should be ~1.0): 1.0000

Max difference in scalar outputs (should be ~0): 0.000000
Max difference in vector outputs (should be ~0): 0.000000


In [ ]:

from torch_geometric.nn import MessagePassing

class GVPConv(MessagePassing):
    """
    One message-passing layer using GVP units for both the message
    function and the node update function.
    """
    def __init__(self, node_dims, edge_dims, out_dims):
        super().__init__(aggr='add', node_dim=0)
        ns, nv = node_dims      # node scalar/vector dims (input)
        es, ev = edge_dims      # edge scalar/vector dims
        os_, ov = out_dims      # output scalar/vector dims

        # Message GVP: takes concatenated [s_i, s_j, edge_s] and [v_i, v_j, edge_v]
        self.message_gvp = GVP(
            in_dims=(2*ns + es, 2*nv + ev),
            out_dims=(os_, ov)
        )

        # Update GVP: takes [s_i, message_s] and [v_i, message_v], produces new node features
        self.update_gvp = GVP(
            in_dims=(ns + os_, nv + ov),
            out_dims=(os_, ov)
        )

    def forward(self, s, v, edge_index, edge_s, edge_v):
        # propagate triggers message() then aggregate (sum) then we handle update ourselves
        msg_s, msg_v = self.propagate(edge_index, s=s, v=v, edge_s=edge_s, edge_v=edge_v)

        s_cat = torch.cat([s, msg_s], dim=-1)
        v_cat = torch.cat([v, msg_v], dim=-2)
        s_new, v_new = self.update_gvp(s_cat, v_cat)

        return s_new, v_new

    def message(self, s_i, s_j, v_i, v_j, edge_s, edge_v):
        s_in = torch.cat([s_i, s_j, edge_s], dim=-1)
        v_in = torch.cat([v_i, v_j, edge_v.unsqueeze(-2)], dim=-2)
        return self.message_gvp(s_in, v_in)

    def aggregate(self, inputs, index, ptr=None, dim_size=None):
        msg_s, msg_v = inputs
        agg_s = super().aggregate(msg_s, index, ptr, dim_size)
        # Vector aggregation needs the same sum, but over the extra (vo, 3) dims
        agg_v = super().aggregate(msg_v.reshape(msg_v.shape[0], -1), index, ptr, dim_size)
        agg_v = agg_v.reshape(dim_size, msg_v.shape[1], 3)
        return agg_s, agg_v

In [ ]:
# Build a tiny synthetic graph: 5 nodes, a few edges
N = 5
edge_index = torch.tensor([[0, 1, 2, 3], [1, 2, 3, 4]], dtype=torch.long)  # simple chain
edge_s = torch.randn(4, 25)   # matches your 25-dim edge_attr
edge_v = torch.randn(4, 3)    # matches your 3-dim edge_vector

s = torch.randn(N, 128)
v = torch.randn(N, 4, 3)

conv = GVPConv(node_dims=(128, 4), edge_dims=(25, 1), out_dims=(128, 16))
s_out, v_out = conv(s, v, edge_index, edge_s, edge_v)

print(f"s_out shape: {s_out.shape}  (expect (5, 128))")
print(f"v_out shape: {v_out.shape}  (expect (5, 16, 3))")

s_out shape: torch.Size([5, 128])  (expect (5, 128))
v_out shape: torch.Size([5, 16, 3])  (expect (5, 16, 3))


In [ ]:
torch.manual_seed(7)
R = random_rotation_matrix()

conv.eval()
with torch.no_grad():
    s_orig, v_orig = conv(s, v, edge_index, edge_s, edge_v)

    v_rotated = v @ R.T
    edge_v_rotated = edge_v @ R.T   # edge vectors must rotate too, consistently

    s_rot, v_rot_output = conv(s, v_rotated, edge_index, edge_s, edge_v_rotated)

    v_orig_then_rotated = v_orig @ R.T

    scalar_diff = (s_orig - s_rot).abs().max().item()
    vector_diff = (v_orig_then_rotated - v_rot_output).abs().max().item()

print(f"Max difference in scalar outputs (should be ~0): {scalar_diff:.6f}")
print(f"Max difference in vector outputs (should be ~0): {vector_diff:.6f}")

Max difference in scalar outputs (should be ~0): 0.000000
Max difference in vector outputs (should be ~0): 0.000000


important

In [ ]:
class GVPEncoder(nn.Module):
    def __init__(self, node_in_dim=517, edge_in_dim=25, hidden_s=128, hidden_v=16, n_layers=3):
        super().__init__()

        # Project raw scalar features down to hidden_s
        self.input_proj = nn.Sequential(
            nn.Linear(node_in_dim, hidden_s),
            nn.LayerNorm(hidden_s)
        )

        # 3 GVPConv layers; vector channels grow 4 -> 16 -> 16 as specified
        self.layer1 = GVPConv(node_dims=(hidden_s, 4),  edge_dims=(edge_in_dim, 1), out_dims=(hidden_s, hidden_v))
        self.layer2 = GVPConv(node_dims=(hidden_s, hidden_v), edge_dims=(edge_in_dim, 1), out_dims=(hidden_s*2, hidden_v))
        self.layer3 = GVPConv(node_dims=(hidden_s*2, hidden_v), edge_dims=(edge_in_dim, 1), out_dims=(hidden_s*2, hidden_v))

    def compute_backbone_vectors(self, pos, edge_index, edge_type, n_nodes):
        """
        Derives 4 initial vector features per node from backbone geometry:
        direction to next residue, direction from previous residue,
        and two more derived directions -- using only backbone (type 0) edges.
        Falls back to zero vectors at chain termini where a neighbor is missing.
        """
        backbone_mask = edge_type == 0
        bb_edge_index = edge_index[:, backbone_mask]

        v_init = torch.zeros(n_nodes, 4, 3, device=pos.device)

        # direction to next residue (i -> i+1) and from previous (i-1 -> i)
        src, dst = bb_edge_index[0], bb_edge_index[1]
        forward_mask = dst == src + 1
        backward_mask = dst == src - 1

        fwd_src, fwd_dst = src[forward_mask], dst[forward_mask]
        if len(fwd_src) > 0:
            direction_fwd = pos[fwd_dst] - pos[fwd_src]
            direction_fwd = direction_fwd / (direction_fwd.norm(dim=-1, keepdim=True) + 1e-8)
            v_init[fwd_src, 0] = direction_fwd
            v_init[fwd_dst, 1] = -direction_fwd  # direction back to previous

        # two more channels: same vectors again (placeholder for orientation frame
        # cross-product terms) -- kept simple and stable for this stage
        v_init[:, 2] = v_init[:, 0]
        v_init[:, 3] = v_init[:, 1]

        return v_init

    def forward(self, x, pos, edge_index, edge_attr, edge_vector, edge_type):
        s = self.input_proj(x)
        v = self.compute_backbone_vectors(pos, edge_index, edge_type, x.shape[0])

        s1, v1 = self.layer1(s, v, edge_index, edge_attr, edge_vector)
        s1 = s1 + s if s1.shape == s.shape else s1   # residual where shapes allow

        s2, v2 = self.layer2(s1, v1, edge_index, edge_attr, edge_vector)
        s3, v3 = self.layer3(s2, v2, edge_index, edge_attr, edge_vector)

        return s3, v3   # final per-residue scalar + vector features


# Test on the real graph we loaded earlier
encoder = GVPEncoder()
encoder.eval()

with torch.no_grad():
    s_out, v_out = encoder(
        sample_graph.x, sample_graph.pos, sample_graph.edge_index,
        sample_graph.edge_attr, sample_graph.edge_vector, sample_graph.edge_type
    )

print(f"Input residues: {sample_graph.x.shape[0]}")
print(f"Output s shape: {s_out.shape}  (expect (n_residues, 256))")
print(f"Output v shape: {v_out.shape}  (expect (n_residues, 16, 3))")

Input residues: 1139
Output s shape: torch.Size([1139, 256])  (expect (n_residues, 256))
Output v shape: torch.Size([1139, 16, 3])  (expect (n_residues, 16, 3))


In [ ]:
# Verify pLDDT is at index 25 in the raw node features
print(f"Sample protein source: {sample_graph.source}")
print(f"pLDDT column (index 25) stats: min={sample_graph.x[:,25].min():.3f}, "
      f"max={sample_graph.x[:,25].max():.3f}, mean={sample_graph.x[:,25].mean():.3f}")

# Cross check with a known AF2/ESMFold protein where pLDDT should vary,
# not just be a constant 1.0 (which is what PDB proteins show)
test_uid_2 = pairs_final_aligned[pairs_final_aligned['protein1'] != sample_graph.uniprot_id]['protein1'].iloc[0]
test_graph_2 = torch.load(f'{graph_dir}/{test_uid_2}.pt', weights_only=False)
print(f"\nSecond protein: {test_uid_2}, source={test_graph_2.source}")
print(f"pLDDT column stats: min={test_graph_2.x[:,25].min():.3f}, "
      f"max={test_graph_2.x[:,25].max():.3f}, mean={test_graph_2.x[:,25].mean():.3f}")

Sample protein source: PDB
pLDDT column (index 25) stats: min=1.000, max=1.000, mean=1.000

Second protein: Q16654, source=PDB
pLDDT column stats: min=1.000, max=1.000, mean=1.000


In [ ]:
# Find one AF2 and one ESMFold protein from the actual pairs
import torch

af2_uid = None
esm_uid = None

for uid in pairs_final_aligned['protein1'].tolist() + pairs_final_aligned['protein2'].tolist():
    g = torch.load(f'{graph_dir}/{uid}.pt', weights_only=False)
    if g.source == 'AF2' and af2_uid is None:
        af2_uid = uid
    elif g.source == 'ESMFold' and esm_uid is None:
        esm_uid = uid
    if af2_uid and esm_uid:
        break

print(f"AF2 example: {af2_uid}")
print(f"ESMFold example: {esm_uid}")

AF2 example: E9PNW1
ESMFold example: Q6PHR5


In [ ]:
# Check pLDDT column on each
for uid, label in [(af2_uid, 'AF2'), (esm_uid, 'ESMFold')]:
    g = torch.load(f'{graph_dir}/{uid}.pt', weights_only=False)
    plddt_col = g.x[:, 25]
    print(f"{label} ({uid}): min={plddt_col.min():.3f}, max={plddt_col.max():.3f}, mean={plddt_col.mean():.3f}")

AF2 (E9PNW1): min=0.281, max=0.983, mean=0.811
ESMFold (Q6PHR5): min=0.370, max=0.980, mean=0.786


In [ ]:
class ConfidenceGatedPooling(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        # Takes [residue_embedding, plddt] -> attention logit
        self.gate_mlp = nn.Sequential(
            nn.Linear(hidden_dim + 1, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, s, plddt, batch_index=None):
        """
        s: (N, hidden_dim) per-residue scalar features from the encoder
        plddt: (N,) per-residue confidence, already 0-1 normalized
        batch_index: (N,) which protein each residue belongs to, for batched
                     graphs (PyG convention) -- None means single protein
        Returns: (hidden_dim,) or (B, hidden_dim) pooled protein embedding(s)
        """
        gate_input = torch.cat([s, plddt.unsqueeze(-1)], dim=-1)
        attn_logits = self.gate_mlp(gate_input).squeeze(-1)   # (N,)

        # Combine learned attention with the raw pLDDT confidence directly --
        # this is the actual "gating": low pLDDT residues get pulled toward
        # zero weight regardless of what the learned attention says
        gated_logits = attn_logits + torch.log(plddt + 1e-8)

        if batch_index is None:
            weights = torch.softmax(gated_logits, dim=0)        # (N,)
            pooled = (weights.unsqueeze(-1) * s).sum(dim=0)      # (hidden_dim,)
            return pooled, weights
        else:
            from torch_geometric.utils import softmax as pyg_softmax
            weights = pyg_softmax(gated_logits, batch_index)     # (N,) softmax per-graph
            weighted = weights.unsqueeze(-1) * s
            from torch_geometric.utils import scatter
            pooled = scatter(weighted, batch_index, dim=0, reduce='sum')  # (B, hidden_dim)
            return pooled, weights


# Test on our single real protein first (no batching yet)
pooling = ConfidenceGatedPooling(hidden_dim=256)
pooling.eval()

plddt_values = sample_graph.x[:, 25]  # constant 1.0 for this PDB protein

with torch.no_grad():
    pooled_emb, attn_weights = pooling(s_out, plddt_values)

print(f"Pooled embedding shape: {pooled_emb.shape}  (expect (256,))")
print(f"Attention weights shape: {attn_weights.shape}  (expect ({sample_graph.x.shape[0]},))")
print(f"Attention weights sum to 1.0: {attn_weights.sum().item():.4f}")

Pooled embedding shape: torch.Size([256])  (expect (256,))
Attention weights shape: torch.Size([1139])  (expect (1139,))
Attention weights sum to 1.0: 1.0000


In [ ]:
# Test gating behavior on the AF2 protein (E9PNW1), which has real pLDDT variation
af2_graph = torch.load(f'{graph_dir}/{af2_uid}.pt', weights_only=False)

with torch.no_grad():
    s_af2, v_af2 = encoder(
        af2_graph.x, af2_graph.pos, af2_graph.edge_index,
        af2_graph.edge_attr, af2_graph.edge_vector, af2_graph.edge_type
    )
    plddt_af2 = af2_graph.x[:, 25]
    pooled_af2, weights_af2 = pooling(s_af2, plddt_af2)

# Correlation between pLDDT and attention weight -- should be positive
# (higher confidence -> higher weight)
import numpy as np
corr = np.corrcoef(plddt_af2.numpy(), weights_af2.numpy())[0,1]
print(f"Correlation between pLDDT and attention weight: {corr:.3f}")
print(f"(Should be positive -- low pLDDT residues should get low weight)")

low_conf_mask = plddt_af2 < 0.5
high_conf_mask = plddt_af2 > 0.9
print(f"\nMean weight for low-confidence residues (pLDDT<0.5): {weights_af2[low_conf_mask].mean().item():.6f}")
print(f"Mean weight for high-confidence residues (pLDDT>0.9): {weights_af2[high_conf_mask].mean().item():.6f}")

Correlation between pLDDT and attention weight: 0.557
(Should be positive -- low pLDDT residues should get low weight)

Mean weight for low-confidence residues (pLDDT<0.5): 0.001727
Mean weight for high-confidence residues (pLDDT>0.9): 0.003252


In [ ]:
class InteractionHead(nn.Module):
    def __init__(self, embed_dim=256, hidden1=512, hidden2=128, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 4, hidden1),
            nn.LayerNorm(hidden1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.LayerNorm(hidden2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1)
        )

    def forward(self, h_a, h_b):
        diff = torch.abs(h_a - h_b)
        prod = h_a * h_b
        combined = torch.cat([h_a, h_b, diff, prod], dim=-1)
        logit = self.mlp(combined)
        return logit.squeeze(-1)


# Test using our two real pooled embeddings (P13569 and E9PNW1)
# Note: these two aren't necessarily a real pair in your dataset --
# this is purely a shape/wiring test, not a meaningful prediction
head = InteractionHead(embed_dim=256)
head.eval()

with torch.no_grad():
    logit = head(pooled_emb.unsqueeze(0), pooled_af2.unsqueeze(0))
    prob = torch.sigmoid(logit)

print(f"Logit: {logit.item():.4f}")
print(f"Probability: {prob.item():.4f}")

Logit: 0.0220
Probability: 0.5055


In [ ]:
class PPIModel(nn.Module):
    def __init__(self, node_in_dim=517, edge_in_dim=25, hidden_s=128, hidden_v=16):
        super().__init__()
        self.encoder = GVPEncoder(node_in_dim=node_in_dim, edge_in_dim=edge_in_dim,
                                    hidden_s=hidden_s, hidden_v=hidden_v)
        # Encoder output scalar dim is hidden_s*2 (grows in layer2/3), matching pooling's expectation
        self.pooling = ConfidenceGatedPooling(hidden_dim=hidden_s*2)
        self.head = InteractionHead(embed_dim=hidden_s*2)

    def encode_protein(self, graph, batch_index=None):
        s_out, v_out = self.encoder(
            graph.x, graph.pos, graph.edge_index,
            graph.edge_attr, graph.edge_vector, graph.edge_type
        )
        plddt = graph.x[:, 25]
        pooled, _ = self.pooling(s_out, plddt, batch_index=batch_index)
        return pooled

    def forward(self, graph_a, graph_b, batch_a=None, batch_b=None):
        h_a = self.encode_protein(graph_a, batch_a)
        h_b = self.encode_protein(graph_b, batch_b)

        if batch_a is None:
            h_a = h_a.unsqueeze(0)
            h_b = h_b.unsqueeze(0)

        logit = self.head(h_a, h_b)
        return logit


# Full end-to-end test: one real pair, single (un-batched) graphs
model = PPIModel()
model.eval()

with torch.no_grad():
    logit = model(sample_graph, af2_graph)
    prob = torch.sigmoid(logit)

print(f"Full model test (single pair, unbatched):")
print(f"Logit: {logit.item():.4f}")
print(f"Probability: {prob.item():.4f}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal model parameters: {total_params:,}")

Full model test (single pair, unbatched):
Logit: -0.4349
Probability: 0.3930

Total model parameters: 1,249,700


In [ ]:
from torch_geometric.data import Batch

# Batch a few real graphs together and check what happens to our custom fields
uids_sample = pairs_final_aligned['protein1'].head(4).tolist()
graphs_sample = [torch.load(f'{graph_dir}/{uid}.pt', weights_only=False) for uid in uids_sample]

batched = Batch.from_data_list(graphs_sample)

print(f"Individual graph sizes: {[g.x.shape[0] for g in graphs_sample]}")
print(f"Batched x shape: {batched.x.shape}")
print(f"Batched edge_index shape: {batched.edge_index.shape}")
print(f"Batched edge_vector shape: {batched.edge_vector.shape}")
print(f"Batched pos shape: {batched.pos.shape}")
print(f"\nbatch attribute (which graph each node belongs to): {batched.batch.shape}")
print(f"Unique batch indices: {batched.batch.unique()}")

Individual graph sizes: [1139, 1139, 1139, 1139]
Batched x shape: torch.Size([4556, 517])
Batched edge_index shape: torch.Size([2, 24508])
Batched edge_vector shape: torch.Size([24508, 3])
Batched pos shape: torch.Size([4556, 3])

batch attribute (which graph each node belongs to): torch.Size([4556])
Unique batch indices: tensor([0, 1, 2, 3])


In [ ]:
# Check: are these actually 4 different proteins, or duplicates?
print(f"UIDs sampled: {[g.uniprot_id for g in graphs_sample]}")
print(f"Unique UIDs: {len(set(g.uniprot_id for g in graphs_sample))}")

UIDs sampled: ['P13569', 'P13569', 'P13569', 'P13569']
Unique UIDs: 1


In [ ]:
# Real test of edge_index offsetting: pick edges from graph 2 (batch index 1)
# and confirm they point to node indices within graph 2's actual range,
# not graph 1's range
node_offset_per_graph = [0]
for g in graphs_sample[:-1]:
    node_offset_per_graph.append(node_offset_per_graph[-1] + g.x.shape[0])
print(f"Expected node index ranges per graph: {node_offset_per_graph}")

# Find edges belonging to graph index 1 (second graph) using edge-level batch info
edge_batch = batched.batch[batched.edge_index[0]]  # which graph each edge's source node belongs to
graph1_edges_mask = edge_batch == 1
graph1_edge_index = batched.edge_index[:, graph1_edges_mask]

expected_lo = node_offset_per_graph[1]
expected_hi = node_offset_per_graph[1] + graphs_sample[1].x.shape[0]
actual_min = graph1_edge_index.min().item()
actual_max = graph1_edge_index.max().item()

print(f"\nGraph 1 (second protein) expected node range: [{expected_lo}, {expected_hi})")
print(f"Graph 1 edges actual node index range: [{actual_min}, {actual_max}]")
print(f"Correctly offset: {expected_lo <= actual_min and actual_max < expected_hi}")

Expected node index ranges per graph: [0, 1139, 2278, 3417]

Graph 1 (second protein) expected node range: [1139, 2278)
Graph 1 edges actual node index range: [1139, 2277]
Correctly offset: True


In [ ]:
# Get genuinely distinct proteins this time
unique_uids_sample = pairs_final_aligned['protein1'].drop_duplicates().head(4).tolist()
print(f"Distinct UIDs: {unique_uids_sample}")

graphs_distinct = [torch.load(f'{graph_dir}/{uid}.pt', weights_only=False) for uid in unique_uids_sample]
print(f"Sizes: {[g.x.shape[0] for g in graphs_distinct]}")

batched_distinct = Batch.from_data_list(graphs_distinct)

node_offset = [0]
for g in graphs_distinct[:-1]:
    node_offset.append(node_offset[-1] + g.x.shape[0])

all_correct = True
for graph_idx in range(len(graphs_distinct)):
    edge_batch = batched_distinct.batch[batched_distinct.edge_index[0]]
    mask = edge_batch == graph_idx
    edges = batched_distinct.edge_index[:, mask]
    lo = node_offset[graph_idx]
    hi = lo + graphs_distinct[graph_idx].x.shape[0]
    ok = (edges.min().item() >= lo) and (edges.max().item() < hi)
    all_correct &= ok
    print(f"Graph {graph_idx} ({unique_uids_sample[graph_idx]}, {graphs_distinct[graph_idx].x.shape[0]} residues): "
          f"expected [{lo},{hi}), actual [{edges.min().item()},{edges.max().item()}], correct={ok}")

print(f"\nAll graphs correctly offset: {all_correct}")

Distinct UIDs: ['P13569', 'Q16654', 'E9PNW1', 'P11233']
Sizes: [1139, 337, 317, 173]
Graph 0 (P13569, 1139 residues): expected [0,1139), actual [0,1138], correct=True
Graph 1 (Q16654, 337 residues): expected [1139,1476), actual [1139,1475], correct=True
Graph 2 (E9PNW1, 317 residues): expected [1476,1793), actual [1476,1792], correct=True
Graph 3 (P11233, 173 residues): expected [1793,1966), actual [1793,1965], correct=True

All graphs correctly offset: True


In [ ]:
from torch.utils.data import Dataset, DataLoader

class PPIPairDataset(Dataset):
    def __init__(self, pairs_df, graph_dir):
        self.pairs_df = pairs_df.reset_index(drop=True)
        self.graph_dir = graph_dir

    def __len__(self):
        return len(self.pairs_df)

    def __getitem__(self, idx):
        row = self.pairs_df.iloc[idx]
        graph_a = torch.load(f'{self.graph_dir}/{row["protein1"]}.pt', weights_only=False)
        graph_b = torch.load(f'{self.graph_dir}/{row["protein2"]}.pt', weights_only=False)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return graph_a, graph_b, label


def ppi_collate_fn(batch):
    graphs_a, graphs_b, labels = zip(*batch)
    batch_a = Batch.from_data_list(list(graphs_a))
    batch_b = Batch.from_data_list(list(graphs_b))
    labels = torch.stack(labels)
    return batch_a, batch_b, labels


# Build datasets for each split
train_df = pairs_final_aligned[pairs_final_aligned['split'] == 'train']
val_df = pairs_final_aligned[pairs_final_aligned['split'] == 'val']
test_df = pairs_final_aligned[pairs_final_aligned['split'] == 'test']

train_dataset = PPIPairDataset(train_df, graph_dir)
val_dataset = PPIPairDataset(val_df, graph_dir)
test_dataset = PPIPairDataset(test_df, graph_dir)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=ppi_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=ppi_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=ppi_collate_fn)

# Test: pull one real batch and check shapes
batch_a, batch_b, labels = next(iter(train_loader))
print(f"\nBatch A: {batch_a.x.shape[0]} total residues across {batch_a.batch.max().item()+1} proteins")
print(f"Batch B: {batch_b.x.shape[0]} total residues across {batch_b.batch.max().item()+1} proteins")
print(f"Labels shape: {labels.shape}, values: {labels[:5]}")

Train: 4856, Val: 80, Test: 80

Batch A: 4233 total residues across 16 proteins
Batch B: 5139 total residues across 16 proteins
Labels shape: torch.Size([16]), values: tensor([0., 0., 1., 0., 1.])


In [ ]:
model.eval()
with torch.no_grad():
    logits = model(batch_a, batch_b, batch_a.batch, batch_b.batch)
    probs = torch.sigmoid(logits)

print(f"Logits shape: {logits.shape}  (expect ({len(labels)},))")
print(f"Sample probabilities: {probs[:5]}")

Logits shape: torch.Size([16])  (expect (16,))
Sample probabilities: tensor([0.3836, 0.3908, 0.4024, 0.3911, 0.4529])


In [ ]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef

def evaluate(loader, model, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch_a, batch_b, labels in loader:
            batch_a, batch_b = batch_a.to(device), batch_b.to(device)
            logits = model(batch_a, batch_b, batch_a.batch, batch_b.batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = (all_probs >= 0.5).astype(int)

    return {
        'auroc': roc_auc_score(all_labels, all_probs),
        'aupr': average_precision_score(all_labels, all_probs),
        'f1': f1_score(all_labels, preds),
        'mcc': matthews_corrcoef(all_labels, preds),
    }

In [ ]:
# Re-initialize model and optimizer fresh (important -- the NaN'd weights
# from the crashed run are still in `model`, don't reuse them)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PPIModel().to(device)

n_pos = (train_df['label'] == 1).sum()
n_neg = (train_df['label'] == 0).sum()
pos_weight = torch.tensor([n_neg / n_pos]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

MAX_EPOCHS = 50
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, epochs=MAX_EPOCHS,
    steps_per_epoch=len(train_loader), pct_start=0.05, final_div_factor=100
)

# NO GradScaler, NO autocast -- plain float32 throughout
best_val_aupr = -1
patience = 15
epochs_no_improve = 0
history = []

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0
    nan_batches = 0

    for batch_a, batch_b, labels in train_loader:
        batch_a, batch_b, labels = batch_a.to(device), batch_b.to(device), labels.to(device)
        optimizer.zero_grad()

        logits = model(batch_a, batch_b, batch_a.batch, batch_b.batch)
        loss = criterion(logits, labels)

        if torch.isnan(loss):
            nan_batches += 1
            continue   # skip this batch rather than corrupting the whole model

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / max(len(train_loader) - nan_batches, 1)
    val_metrics = evaluate(val_loader, model, device)
    history.append({'epoch': epoch, 'train_loss': avg_loss, **val_metrics})

    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | loss={avg_loss:.4f} | nan_batches={nan_batches} | "
          f"val_auroc={val_metrics['auroc']:.4f} | val_aupr={val_metrics['aupr']:.4f} | "
          f"val_f1={val_metrics['f1']:.4f} | val_mcc={val_metrics['mcc']:.4f}")

    if val_metrics['aupr'] > best_val_aupr:
        best_val_aupr = val_metrics['aupr']
        epochs_no_improve = 0
        torch.save(model.state_dict(), f'{base_dir}/data/processed/best_model.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

print(f"\nBest validation AUPR: {best_val_aupr:.4f}")

Epoch 1/50 | loss=0.9642 | nan_batches=0 | val_auroc=0.8479 | val_aupr=0.8081 | val_f1=0.6538 | val_mcc=0.4977
Epoch 2/50 | loss=0.8633 | nan_batches=0 | val_auroc=0.6788 | val_aupr=0.5134 | val_f1=0.1714 | val_mcc=0.0814
Epoch 3/50 | loss=0.8162 | nan_batches=0 | val_auroc=0.4956 | val_aupr=0.3982 | val_f1=0.3721 | val_mcc=0.2002
Epoch 4/50 | loss=0.7696 | nan_batches=0 | val_auroc=0.4571 | val_aupr=0.3370 | val_f1=0.0000 | val_mcc=-0.1488
Epoch 5/50 | loss=0.7296 | nan_batches=0 | val_auroc=0.4888 | val_aupr=0.3792 | val_f1=0.0541 | val_mcc=-0.1647
Epoch 6/50 | loss=0.7065 | nan_batches=0 | val_auroc=0.3753 | val_aupr=0.3400 | val_f1=0.1143 | val_mcc=-0.0173
Epoch 7/50 | loss=0.6807 | nan_batches=0 | val_auroc=0.3874 | val_aupr=0.3072 | val_f1=0.0588 | val_mcc=-0.0873
Epoch 8/50 | loss=0.6520 | nan_batches=0 | val_auroc=0.4963 | val_aupr=0.3624 | val_f1=0.0541 | val_mcc=-0.1647
Epoch 9/50 | loss=0.6350 | nan_batches=0 | val_auroc=0.4632 | val_aupr=0.3793 | val_f1=0.3043 | val_mcc=0.0

In [ ]:
# Check training set performance at a few epochs for comparison
train_metrics_check = evaluate(train_loader, model, device)
print(f"Final model (epoch 16) on TRAIN set: {train_metrics_check}")

Final model (epoch 16) on TRAIN set: {'auroc': np.float64(0.9501403092826075), 'aupr': np.float64(0.9022128516956189), 'f1': 0.8413338219127886, 'mcc': np.float64(0.7805707927404425)}


In [ ]:
# Also check: what does the val set's label balance actually look like?
# With 80 pairs total, a handful of label flips changes metrics a lot
print(val_df['label'].value_counts())
print(f"\nVal set positive pairs: {(val_df['label']==1).sum()} out of {len(val_df)}")

label
0    51
1    29
Name: count, dtype: int64

Val set positive pairs: 29 out of 80


In [ ]:
# Rebuild the model with higher dropout in the interaction head
class InteractionHead(nn.Module):
    def __init__(self, embed_dim=256, hidden1=512, hidden2=128, dropout=0.5):  # 0.3 -> 0.5
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim * 4, hidden1),
            nn.LayerNorm(hidden1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2),
            nn.LayerNorm(hidden2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden2, 1)
        )

    def forward(self, h_a, h_b):
        diff = torch.abs(h_a - h_b)
        prod = h_a * h_b
        combined = torch.cat([h_a, h_b, diff, prod], dim=-1)
        logit = self.mlp(combined)
        return logit.squeeze(-1)


# Re-run the PPIModel class definition so it picks up the new InteractionHead default
class PPIModel(nn.Module):
    def __init__(self, node_in_dim=517, edge_in_dim=25, hidden_s=128, hidden_v=16):
        super().__init__()
        self.encoder = GVPEncoder(node_in_dim=node_in_dim, edge_in_dim=edge_in_dim,
                                    hidden_s=hidden_s, hidden_v=hidden_v)
        self.pooling = ConfidenceGatedPooling(hidden_dim=hidden_s*2)
        self.head = InteractionHead(embed_dim=hidden_s*2, dropout=0.5)

    def encode_protein(self, graph, batch_index=None):
        s_out, v_out = self.encoder(
            graph.x, graph.pos, graph.edge_index,
            graph.edge_attr, graph.edge_vector, graph.edge_type
        )
        plddt = graph.x[:, 25]
        pooled, _ = self.pooling(s_out, plddt, batch_index=batch_index)
        return pooled

    def forward(self, graph_a, graph_b, batch_a=None, batch_b=None):
        h_a = self.encode_protein(graph_a, batch_a)
        h_b = self.encode_protein(graph_b, batch_b)
        if batch_a is None:
            h_a = h_a.unsqueeze(0)
            h_b = h_b.unsqueeze(0)
        logit = self.head(h_a, h_b)
        return logit

In [ ]:
# Fresh model + optimizer with lower LR and higher weight decay
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PPIModel().to(device)

n_pos = (train_df['label'] == 1).sum()
n_neg = (train_df['label'] == 0).sum()
pos_weight = torch.tensor([n_neg / n_pos]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)  # lr 1e-3->1e-4, wd 1e-5->1e-3

MAX_EPOCHS = 50
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-4, epochs=MAX_EPOCHS,   # max_lr also lowered
    steps_per_epoch=len(train_loader), pct_start=0.05, final_div_factor=100
)

best_val_aupr = -1
patience = 15
epochs_no_improve = 0
history = []

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0
    nan_batches = 0

    for batch_a, batch_b, labels in train_loader:
        batch_a, batch_b, labels = batch_a.to(device), batch_b.to(device), labels.to(device)
        optimizer.zero_grad()

        logits = model(batch_a, batch_b, batch_a.batch, batch_b.batch)
        loss = criterion(logits, labels)

        if torch.isnan(loss):
            nan_batches += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / max(len(train_loader) - nan_batches, 1)
    val_metrics = evaluate(val_loader, model, device)
    history.append({'epoch': epoch, 'train_loss': avg_loss, **val_metrics})

    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | loss={avg_loss:.4f} | nan_batches={nan_batches} | "
          f"val_auroc={val_metrics['auroc']:.4f} | val_aupr={val_metrics['aupr']:.4f} | "
          f"val_f1={val_metrics['f1']:.4f} | val_mcc={val_metrics['mcc']:.4f}")

    if val_metrics['aupr'] > best_val_aupr:
        best_val_aupr = val_metrics['aupr']
        epochs_no_improve = 0
        torch.save(model.state_dict(), f'{base_dir}/data/processed/best_model.pt')
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

print(f"\nBest validation AUPR: {best_val_aupr:.4f}")

Epoch 1/50 | loss=1.0326 | nan_batches=0 | val_auroc=0.6241 | val_aupr=0.5833 | val_f1=0.5000 | val_mcc=0.0818
Epoch 2/50 | loss=0.9047 | nan_batches=0 | val_auroc=0.8235 | val_aupr=0.7496 | val_f1=0.5652 | val_mcc=0.4346
Epoch 3/50 | loss=0.7976 | nan_batches=0 | val_auroc=0.6748 | val_aupr=0.5974 | val_f1=0.0000 | val_mcc=0.0000
Epoch 4/50 | loss=0.7353 | nan_batches=0 | val_auroc=0.8019 | val_aupr=0.7362 | val_f1=0.2424 | val_mcc=0.3042
Epoch 5/50 | loss=0.6777 | nan_batches=0 | val_auroc=0.6362 | val_aupr=0.5565 | val_f1=0.4615 | val_mcc=0.4226
Epoch 6/50 | loss=0.6326 | nan_batches=0 | val_auroc=0.6200 | val_aupr=0.4968 | val_f1=0.0606 | val_mcc=-0.0537
Epoch 7/50 | loss=0.5960 | nan_batches=0 | val_auroc=0.5220 | val_aupr=0.4591 | val_f1=0.2778 | val_mcc=0.2266
Epoch 8/50 | loss=0.5858 | nan_batches=0 | val_auroc=0.4212 | val_aupr=0.4256 | val_f1=0.3721 | val_mcc=0.2002
Epoch 9/50 | loss=0.5488 | nan_batches=0 | val_auroc=0.5145 | val_aupr=0.4084 | val_f1=0.2632 | val_mcc=0.1430


In [ ]:
import json

with open(f'{base_dir}/data/processed/training_history_run2.json', 'w') as f:
    json.dump(history, f, indent=2, default=float)

print(f"Saved training history: {len(history)} epochs")

Saved training history: 17 epochs


In [ ]:
import random

# Reuse the same cluster assignments from Step 7 (protein_to_cluster)
# If you don't have this loaded in the new notebook, we'll need to reload it
print(f"protein_to_cluster available: {'protein_to_cluster' in dir()}")

protein_to_cluster available: False


In [ ]:
import json

# We need the cluster assignments to do proper protein-level k-fold.
# Check if we saved this in Step 7 -- if not, we'll rebuild from CD-HIT output
with open(f'{base_dir}/data/labels/protein_split_mapping.json', 'r') as f:
    protein_split_original = json.load(f)
print(f"Loaded protein_split_original: {len(protein_split_original)} proteins")

Loaded protein_split_original: 2608 proteins


In [ ]:
import os
print(os.path.exists(f'{base_dir}/data/raw/proteins_clustered.fasta.clstr'))

False


In [ ]:
# Get sequences for the 1,758 proteins -- we have these saved from Step 8D
import pickle

with open(f'{base_dir}/data/processed/aligned_sequences.pkl', 'rb') as f:
    aligned_sequences = pickle.load(f)

print(f"Aligned sequences available: {len(aligned_sequences)}")

# Confirm this covers exactly our working protein set
final_proteins_list = list(set(pairs_final_aligned['protein1']) | set(pairs_final_aligned['protein2']))
print(f"Proteins needed: {len(final_proteins_list)}")
missing = set(final_proteins_list) - set(aligned_sequences.keys())
print(f"Missing sequences: {len(missing)}")

Aligned sequences available: 1851
Proteins needed: 1758
Missing sequences: 0


In [ ]:
# Write FASTA for CD-HIT
with open('/content/proteins_for_kfold.fasta', 'w') as f:
    for uid in final_proteins_list:
        if uid in aligned_sequences:
            f.write(f">{uid}\n{aligned_sequences[uid]}\n")

print("FASTA written")
!wc -l /content/proteins_for_kfold.fasta

FASTA written
3516 /content/proteins_for_kfold.fasta


In [ ]:
!cd-hit -i /content/proteins_for_kfold.fasta -o /content/proteins_kfold_clustered.fasta -c 0.4 -n 2 -d 0 -M 0 -T 0

/bin/bash: line 1: cd-hit: command not found


In [ ]:
!apt-get install -y cd-hit 2>&1 | tail -5

(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../cd-hit_4.8.1-4_amd64.deb ...
Unpacking cd-hit (4.8.1-4) ...
Setting up cd-hit (4.8.1-4) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!which cd-hit

/usr/bin/cd-hit


In [ ]:
!cd-hit -i /content/proteins_for_kfold.fasta -o /content/proteins_kfold_clustered.fasta -c 0.4 -n 2 -d 0 -M 0 -T 0

Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i /content/proteins_for_kfold.fasta -o
         /content/proteins_kfold_clustered.fasta -c 0.4 -n 2 -d
         0 -M 0 -T 0

Started: Mon Jun 29 03:38:14 2026
                            Output                              
----------------------------------------------------------------
total number of CPUs in the system is 2
Actual number of CPUs to be used: 2

total seq: 1758
longest and shortest : 2353 and 21
Total letters: 583050
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 0M
Buffer          : 2 X 11M = 22M
Table           : 2 X 0M = 0M
Miscellaneous   : 0M
Total           : 22M

Table limit with the given memory limit:
Max number of representatives: 2414213
Max number of word counting entries: 166989039

# comparing sequences from          0  to        439
---------- new table with      399 representatives
# comparing sequences from        439  to        768
99.9%-

In [ ]:
!grep -c "^>Cluster" /content/proteins_kfold_clustered.fasta.clstr

1456


In [ ]:
import re

with open('/content/proteins_kfold_clustered.fasta.clstr') as f:
    content = f.read()

protein_to_cluster = {}
clusters_raw = content.split('>Cluster ')[1:]

for c in clusters_raw:
    lines = c.strip().split('\n')
    cluster_id = int(lines[0])
    for line in lines[1:]:
        match = re.search(r'>(\S+?)\.\.\.', line)
        if match:
            uid = match.group(1)
            protein_to_cluster[uid] = cluster_id

print(f"Proteins mapped to clusters: {len(protein_to_cluster):,}")
print(f"Unique cluster IDs: {len(set(protein_to_cluster.values())):,}")

Proteins mapped to clusters: 1,758
Unique cluster IDs: 1,456


In [ ]:
import random

random.seed(42)

all_cluster_ids = list(set(protein_to_cluster.values()))
random.shuffle(all_cluster_ids)

K = 5
fold_clusters = [[] for _ in range(K)]
for i, cid in enumerate(all_cluster_ids):
    fold_clusters[i % K].append(cid)

# Build protein -> fold mapping
protein_fold = {}
for fold_idx, cluster_list in enumerate(fold_clusters):
    cluster_set = set(cluster_list)
    for uid, cid in protein_to_cluster.items():
        if cid in cluster_set:
            protein_fold[uid] = fold_idx

from collections import Counter
fold_protein_counts = Counter(protein_fold.values())
print(f"Proteins per fold: {dict(sorted(fold_protein_counts.items()))}")

Proteins per fold: {0: 362, 1: 341, 2: 341, 3: 363, 4: 351}


In [ ]:
def get_pair_fold(row, protein_fold):
    f1 = protein_fold.get(row['protein1'])
    f2 = protein_fold.get(row['protein2'])
    if f1 == f2:
        return f1
    return -1  # cross-fold pair, must be dropped

pairs_final_aligned['fold'] = pairs_final_aligned.apply(
    lambda row: get_pair_fold(row, protein_fold), axis=1
)

print(f"Total pairs: {len(pairs_final_aligned):,}")
print(f"\nFold distribution (including -1 = cross-fold, to be dropped):")
print(pairs_final_aligned['fold'].value_counts().sort_index())

n_dropped = (pairs_final_aligned['fold'] == -1).sum()
print(f"\nPairs dropped (cross-fold): {n_dropped:,} ({100*n_dropped/len(pairs_final_aligned):.1f}%)")

Total pairs: 5,016

Fold distribution (including -1 = cross-fold, to be dropped):
fold
-1    3959
 0     244
 1     198
 2     179
 3     238
 4     198
Name: count, dtype: int64

Pairs dropped (cross-fold): 3,959 (78.9%)


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils import resample

def bootstrap_metrics(y_true, y_probs, n_bootstrap=1000, seed=42):
    rng = np.random.RandomState(seed)
    aurocs, auprs = [], []

    y_true = np.array(y_true)
    y_probs = np.array(y_probs)

    for _ in range(n_bootstrap):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        y_t, y_p = y_true[idx], y_probs[idx]
        if len(set(y_t)) < 2:  # skip resamples with only one class
            continue
        aurocs.append(roc_auc_score(y_t, y_p))
        auprs.append(average_precision_score(y_t, y_p))

    return {
        'auroc_mean': np.mean(aurocs), 'auroc_ci': (np.percentile(aurocs, 2.5), np.percentile(aurocs, 97.5)),
        'aupr_mean': np.mean(auprs), 'aupr_ci': (np.percentile(auprs, 2.5), np.percentile(auprs, 97.5)),
    }